<a href="https://colab.research.google.com/github/chenxingqiang/s1-cor/blob/main/nb/Qwen3_(4B)-COR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [ ]:
# -*- coding: utf-8 -*-
"""
Qwen3-4B GRPO with Chain of Reward (CoR).

Self-contained script for Colab. Uses CoR rewards:
R = R_ext + λ·R_int + μ·R_improve + ν·R_converge

Requirements:
  pip install unsloth vllm transformers datasets trl

Run in Colab: Upload this file and run all cells / python qwen3_(4b)_cor.py
"""

import re
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

from datasets import load_dataset, Dataset
import pandas as pd
import numpy as np
import torch

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig, GRPOConfig, GRPOTrainer
from vllm import SamplingParams


# =============================================================================
# CoR Reward Logic (inlined for Colab - no external deps)
# =============================================================================

@dataclass
class RewardConfig:
    lambda_intrinsic: float = 1.0
    self_rating_weight: float = 0.2
    calibration_bonus: float = 0.2
    improvement_weight: float = 0.5
    convergence_weight: float = 0.1
    improvement_discount: float = 0.9


def _parse_completion(completion: str) -> Tuple[str, str]:
    """Parse completion into thinking and answer. Supports CoR, Qwen, generic formats."""
    # CoR format: <thinking>...</thinking><answer>...</answer>
    am = re.search(r"<answer>(.*?)</answer>", completion, re.DOTALL)
    if am:
        thinking = completion[: am.start()].strip()
        return thinking, am.group(1).strip()
    if "<|im_end|>" in completion:
        parts = completion.split("<|im_end|>")
        thinking = parts[0].replace("<|im_start|>assistant\n", "").strip()
        return thinking, (parts[1].strip() if len(parts) > 1 else "")
    match = re.search(r"\n[Aa]nswer:\s*(.+?)$", completion, re.DOTALL)
    if match:
        return completion[: match.start()].strip(), match.group(1).strip()
    match = re.search(r"[Tt]he (?:final )?answer is[:\s]+(.+?)$", completion, re.DOTALL)
    if match:
        return completion[: match.start()].strip(), match.group(1).strip()
    lines = completion.strip().split("\n")
    return ("\n".join(lines[:-1]), lines[-1]) if len(lines) > 1 else (completion, completion)


def _compute_intrinsic_simple(text: str) -> float:
    """Simple intrinsic: consistency, completeness, format, self-rating."""
    s = 0.5
    if re.search(r"\[Self-Rating:", text, re.IGNORECASE):
        s += 0.25
    step_count = len(re.findall(r"(?:Step\s+\d+|^\s*\d+[\.\)]\s+)", text, re.MULTILINE))
    if step_count >= 3:
        s += 0.15
    if re.search(r"(?:Therefore|Thus|Hence|So)", text, re.IGNORECASE):
        s += 0.1
    return max(0.0, min(1.0, s))


def _compute_self_rating_reward(text: str, actual_qualities: Dict[str, float], correct: bool) -> float:
    """Self-rating calibration reward."""
    m = re.search(r"\[Self-Rating:\s*([^\]]+)\]", text, re.IGNORECASE)
    if not m:
        return 0.3
    content = m.group(1)
    ratings = {}
    for part in re.split(r"[,;]\s*", content):
        pm = re.match(r"(\w+)\s*[:=]\s*(\d+(?:\.\d+)?)/10", part.strip())
        if pm:
            ratings[pm.group(1).lower()] = float(pm.group(2)) / 10.0
    if not ratings:
        return 0.5
    cals = [
        1.0 - abs(ratings.get(d, 0.5) - actual_qualities.get(d, 0.5))
        for d in ["consistency", "completeness", "accuracy", "clarity"]
    ]
    cal = sum(cals) / len(cals) if cals else 0.5
    avg = sum(ratings.values()) / len(ratings)
    correctness_cal = avg if correct else (1.0 - avg)
    return 0.4 * cal + 0.3 * correctness_cal + 0.3 * min(1.0, len(ratings) / 4)


def _compute_quality(chain: str) -> float:
    """Q(c) for improvement/convergence."""
    intrinsic = _compute_intrinsic_simple(chain)
    dims = {"consistency": 0.5, "completeness": 0.5, "accuracy": 0.5, "clarity": 0.5}
    self_r = _compute_self_rating_reward(chain, dims, False)
    return 0.8 * intrinsic + 0.2 * self_r


class CoRRewardCalculator:
    """Standalone CoR reward calculator for Colab."""

    def __init__(self, config: RewardConfig):
        self.config = config

    def calculate_external_reward(self, answer: str, gt: str) -> float:
        if not answer or not gt:
            return 0.0
        a = answer.strip().lower().rstrip(".,;:!?")
        g = gt.strip().lower().rstrip(".,;:!?")
        for p in ["the answer is", "answer:", "final answer:"]:
            if a.startswith(p):
                a = a[len(p) :].strip()
        return 1.0 if a == g else 0.0

    def calculate_intrinsic_reward(
        self, thinking: str, include_self_rating: bool = True, final_answer_correct: bool = False
    ) -> Tuple[float, Dict]:
        intrinsic = _compute_intrinsic_simple(thinking)
        dims = {"consistency": 0.5, "completeness": 0.5, "accuracy": 0.5, "clarity": 0.5, "format": 0.5}
        self_r = _compute_self_rating_reward(thinking, dims, final_answer_correct) if include_self_rating else 0.5
        total = (intrinsic + self.config.self_rating_weight * self_r) / (1.0 + self.config.self_rating_weight)
        return total, {"intrinsic": intrinsic, "self_rating": self_r}

    def calculate_total_reward(self, thinking: str, answer: str, gt: str) -> float:
        ext = self.calculate_external_reward(answer, gt)
        intr, _ = self.calculate_intrinsic_reward(thinking, True, ext > 0.5)
        return ext + self.config.lambda_intrinsic * intr

    def compute_cumulative_improvement(self, chains: List[str]) -> float:
        if len(chains) < 2:
            return 0.0
        total = 0.0
        for k in range(len(chains) - 1):
            total += (self.config.improvement_discount ** k) * (
                _compute_quality(chains[k + 1]) - _compute_quality(chains[k])
            )
        return total

    def compute_convergence_reward(self, old_chain: str, new_chain: str) -> float:
        div = abs(_compute_quality(new_chain) - _compute_quality(old_chain))
        return max(0.0, 1.0 - div)

    def calculate_reflection_reward(
        self, chains: List[str], final_answer: str, gt: str
    ) -> float:
        if not chains:
            return 0.0
        ext = self.calculate_external_reward(final_answer, gt)
        intr, _ = self.calculate_intrinsic_reward(chains[-1], True, ext > 0.5)
        imp = self.compute_cumulative_improvement(chains) if len(chains) > 1 else 0.0
        conv = self.compute_convergence_reward(chains[-2], chains[-1]) if len(chains) > 1 else 0.0
        return ext + self.config.lambda_intrinsic * intr + self.config.improvement_weight * imp + self.config.convergence_weight * conv


# =============================================================================
# CoR Format (design.md)
# =============================================================================
REASONING_START = "<thinking>"
REASONING_END = "</thinking>"
ANSWER_START = "<answer>"
ANSWER_END = "</answer>"

SYSTEM_PROMPT = """You are a helpful assistant. Solve the problem step by step.

1. Think through the problem in <thinking>...</thinking>.
2. At the end of your thinking, add a self-rating: [Self-Rating: Consistency=X/10, Completeness=Y/10, Accuracy=Z/10, Clarity=W/10, Format=Z/10]
   Rate each dimension 1-10 (Consistency=logic, Completeness=steps, Accuracy=correctness, Clarity=readability, Format=structure).
3. Give your final answer in <answer>...</answer>."""

# =============================================================================
# CoR Reward Function
# =============================================================================
COR_CONFIG = RewardConfig(
    lambda_intrinsic=1.0,
    self_rating_weight=0.2,
    calibration_bonus=0.2,
    improvement_weight=0.5,
    convergence_weight=0.1,
)
COR_CALCULATOR = CoRRewardCalculator(COR_CONFIG)


def _parse_cor_completion(response: str) -> tuple:
    """Parse CoR completion into thinking and answer.

    Handles: <thinking>...</thinking><answer>...</answer>
    """
    response = response.strip()
    # Remove leading prepended token if present
    if response.startswith(REASONING_START):
        response = response[len(REASONING_START):].strip()

    # Try CoR format
    think_match = re.search(
        rf"{re.escape(REASONING_START)}(.*?){re.escape(REASONING_END)}",
        response, re.DOTALL
    )
    answer_match = re.search(
        rf"{re.escape(ANSWER_START)}(.*?){re.escape(ANSWER_END)}",
        response, re.DOTALL
    )

    if think_match and answer_match:
        thinking = think_match.group(1).strip()
        answer = answer_match.group(1).strip()
        return thinking, answer

    # Partial: has answer but no thinking tags
    if answer_match:
        thinking = response[: answer_match.start()].strip()
        return thinking, answer_match.group(1).strip()

    # Fallback: generic parse
    return _parse_completion(response)


def _extract_reflection_rounds(completion: str) -> list:
    """Extract reflection rounds from completion (design.md Section 7.2).

    Parses [Round 1]...[Reflection]...[Round 2]... format.
    Returns list of chain strings [c_0, c_1, ..., c_K].
    """
    round_pattern = r"\[Round (\d+)\]"
    rounds = list(re.finditer(round_pattern, completion))

    if len(rounds) < 2:
        return [completion]

    chain_sequence = []
    for i, match in enumerate(rounds):
        start = match.end()
        end = rounds[i + 1].start() if i + 1 < len(rounds) else len(completion)
        round_content = completion[start:end].strip()
        # Remove [Reflection] section
        ref_match = re.search(r"\[Reflection\].*$", round_content, re.DOTALL)
        if ref_match:
            round_content = round_content[: ref_match.start()].strip()
        if round_content:
            chain_sequence.append(round_content)

    return chain_sequence if chain_sequence else [completion]


def _extract_ground_truth(example: dict) -> str:
    """Extract expected answer from s1-cor dataset example."""
    # Try boxed format
    for field in ["solution", "attempt", "text", "text_cor"]:
        text = example.get(field, "")
        if not text:
            continue
        match = re.search(r'\\(?:\\\\boxed|boxed)\\{([^}]+)\\}', text)
        if match:
            return match.group(1).strip()
        match = re.search(r'\\boxed\{([^}]+)\}', text)
        if match:
            return match.group(1).strip()

    # Try "Final Answer:" format
    attempt = example.get("attempt", "")
    if attempt:
        m = re.search(r'Final Answer[:\s]+(.+?)(?:\n|$)', attempt, re.IGNORECASE | re.DOTALL)
        if m:
            return m.group(1).strip()

    return str(example.get("expected_answer", ""))


def cor_reward_for_grpo(prompts, completions, answer, **kwargs) -> list:
    """CoR reward function for TRL GRPOTrainer.

    Full formula (design.md Section 2.1):
    R = R_ext + λ·R_int + μ·R_improve + ν·R_converge

    - Single-round: R_improve=0, R_converge=0
    - Multi-round: uses calculate_reflection_reward for all 4 components
    """
    scores = []
    for i, completion in enumerate(completions):
        response = completion[0]["content"]
        gt = answer[i] if i < len(answer) else None

        chain_sequence = _extract_reflection_rounds(response)

        if len(chain_sequence) > 1:
            # Multi-round reflection: full CoR formula with R_improve, R_converge
            thinking, answer_text = _parse_cor_completion(chain_sequence[-1])
            if gt is None:
                intrinsic, _ = COR_CALCULATOR.calculate_intrinsic_reward(
                    chain_sequence[-1], include_self_rating=True, final_answer_correct=False
                )
                improvement = COR_CALCULATOR.improvement_calculator.compute_cumulative_improvement(
                    chain_sequence
                )
                total = intrinsic + COR_CONFIG.improvement_weight * improvement
                scores.append(total)
            else:
                output = COR_CALCULATOR.calculate_reflection_reward(
                    chain_sequence, answer_text, gt
                )
                scores.append(output.total_reward)
        else:
            # Single-round: R_ext + λ·R_int
            thinking, answer_text = _parse_cor_completion(response)
            if gt is None:
                intrinsic, _ = COR_CALCULATOR.calculate_intrinsic_reward(
                    thinking, include_self_rating=True, final_answer_correct=False
                )
                scores.append(intrinsic)
            else:
                output = COR_CALCULATOR.calculate_total_reward(thinking, answer_text, gt)
                scores.append(output.total_reward)

    return scores


# =============================================================================
# Main
# =============================================================================
def main():
    max_seq_length = 2048
    lora_rank = 32

    print("Loading Qwen3-4B...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3-4B-Base",
        max_seq_length=max_seq_length,
        load_in_4bit=False,
        fast_inference=True,
        max_lora_rank=lora_rank,
        gpu_memory_utilization=0.9,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=lora_rank,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=lora_rank * 2,
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )

    # Chat template for CoR
    _sys = SYSTEM_PROMPT.replace("\\", "\\\\").replace("'", "\\'").replace("\n", "\\n")
    chat_template = (
        "{% if messages[0]['role'] == 'system' %}"
        "{{ messages[0]['content'] + eos_token }}"
        "{% set loop_messages = messages[1:] %}"
        "{% else %}"
        "{{ '" + _sys + "' + eos_token }}"
        "{% set loop_messages = messages %}"
        "{% endif %}"
        "{% for message in loop_messages %}"
        "{% if message['role'] == 'user' %}"
        "{{ message['content'] }}"
        "{% elif message['role'] == 'assistant' %}"
        "{{ message['content'] + eos_token }}"
        "{% endif %}"
        "{% endfor %}"
        "{% if add_generation_prompt %}{{ '" + REASONING_START + "' }}"
        "{% endif %}"
    )
    tokenizer.chat_template = chat_template

    # ========== Pre-finetuning (format priming) ==========
    print("Loading pre-finetuning data (OpenMathReasoning-mini)...")
    pre_dataset = load_dataset("unsloth/OpenMathReasoning-mini", split="cot")
    pre_df = pre_dataset.to_pandas()[["expected_answer", "problem", "generated_solution"]]
    is_number = pd.to_numeric(pre_df["expected_answer"], errors="coerce").notnull()
    pre_df = pre_df.iloc[np.where(is_number)[0]]

    def format_pre(x):
        thoughts = x["generated_solution"].replace("<think>", "").replace("</think>", "").strip()
        # Add placeholder self-rating so model learns CoR format
        if "[Self-Rating:" not in thoughts:
            thoughts += "\n[Self-Rating: Consistency=7/10, Completeness=7/10, Accuracy=7/10, Clarity=7/10, Format=7/10]"
        final = f"{REASONING_START}{thoughts}{REASONING_END}{ANSWER_START}{x['expected_answer']}{ANSWER_END}"
        return [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": x["problem"]},
            {"role": "assistant", "content": final},
        ]

    pre_df["Messages"] = pre_df.apply(format_pre, axis=1)
    pre_df["N"] = pre_df["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))
    pre_df = pre_df.loc[pre_df["N"] <= max_seq_length // 2].copy()
    pre_df["text"] = tokenizer.apply_chat_template(pre_df["Messages"].values.tolist(), tokenize=False)
    pre_dataset = Dataset.from_pandas(pre_df)

    print("Pre-finetuning for CoR format...")
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=pre_dataset,
        args=SFTConfig(
            dataset_text_field="text",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=1,
            warmup_steps=5,
            num_train_epochs=2,
            learning_rate=2e-4,
            logging_steps=5,
            optim="adamw_8bit",
            weight_decay=0.001,
            lr_scheduler_type="linear",
            seed=3407,
            report_to="none",
        ),
    )
    trainer.train()

    del pre_dataset
    torch.cuda.empty_cache()
    import gc
    gc.collect()

    # ========== GRPO Data (CoR dataset) ==========
    print("Loading CoR dataset (xingqiang/s1K-cor-deepseek)...")
    try:
        dataset = load_dataset("xingqiang/s1K-cor-deepseek", split="train")
    except Exception:
        print("HF dataset not found, using DAPO-Math-17k...")
        dataset = load_dataset("open-r1/DAPO-Math-17k-Processed", "en", split="train")
        dataset = dataset.select(range(min(1000, len(dataset))))

    def map_to_cor_format(x):
        gt = _extract_ground_truth(x) if isinstance(x, dict) else ""
        prompt = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": x.get("question", x.get("prompt", ""))},
        ]
        return {"prompt": prompt, "answer": gt}

    dataset = dataset.map(map_to_cor_format, remove_columns=dataset.column_names)

    # Filter by prompt length
    tokenized = dataset.map(
        lambda x: {"tokens": tokenizer.apply_chat_template(x["prompt"], add_generation_prompt=True, tokenize=True)},
        batched=True,
    )
    tokenized = tokenized.map(lambda x: {"L": len(x["tokens"])})
    max_len = int(np.quantile(tokenized["L"], 0.9))
    dataset = dataset.select(np.where(np.array(tokenized["L"]) <= max_len)[0])
    del tokenized

    max_prompt_length = max_len + 1
    max_completion_length = max_seq_length - max_prompt_length

    # ========== GRPO Training ==========
    vllm_params = SamplingParams(
        min_p=0.1, top_p=1.0, top_k=-1, seed=3407,
        stop=[tokenizer.eos_token], include_stop_str_in_output=True,
    )

    grpo_args = GRPOConfig(
        vllm_sampling_params=vllm_params,
        temperature=1.0,
        learning_rate=5e-6,
        weight_decay=0.001,
        warmup_ratio=0.1,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        logging_steps=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_generations=8,  # N=8 per design.md
        max_prompt_length=max_prompt_length,
        max_completion_length=max_completion_length,
        max_steps=100,
        save_steps=100,
        report_to="none",
        output_dir="outputs/cor_qwen3_4b",
    )

    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[cor_reward_for_grpo],
        args=grpo_args,
        train_dataset=dataset,
    )
    trainer.train()

    # Save locally (LoRA adapters)
    lora_dir = "grpo_cor_saved_lora"
    model.save_lora(lora_dir)
    tokenizer.save_pretrained(lora_dir)
    print(f"Saved LoRA to {lora_dir}/")

    # Push to HuggingFace Hub (xingqiang account)
    hub_model_id = "xingqiang/cor-grpo-qwen3-4b"
    print(f"Pushing to HuggingFace Hub: {hub_model_id}")
    try:
        model.push_to_hub(hub_model_id, private=True, commit_message="CoR-GRPO Qwen3-4B")
        tokenizer.push_to_hub(hub_model_id, private=True)
        print(f"Model uploaded to: https://huggingface.co/{hub_model_id}")
    except Exception as e:
        print(f"Failed to push to hub: {e}")


# if __name__ == "__main__":
#     main()
